# A2 RNN's full model

# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [27]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell, Input, Concatenate, Lambda, Attention
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

### Stuff

In [28]:
from scipy.ndimage import rotate
tf.random.set_seed(42)

# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [29]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


In [30]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


In [31]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


### More stuff

In [32]:
import tensorflow as tf
import numpy as np

# 1. THE ARITHMETIC LOGIC (Must be defined for the layer to work)
def scheduled_mask_layer(x, training=None, prob=None):
    """
    Applies a Bernoulli mask to the input tensor for Scheduled Sampling.
    Reference: Bengio et al. (2015).
    """
    if training is None:
        training = tf.keras.backend.learning_phase()

    # Create a Bernoulli mask: 1 with probability 'prob', 0 otherwise
    # mask ~ Bernoulli(p)
    random_tensor = tf.random.uniform(tf.shape(x)[:2], minval=0, maxval=1, dtype=tf.float32)
    mask = tf.cast(random_tensor < prob, tf.float32)
    mask = tf.expand_dims(mask, axis=-1) # Align with (Batch, Time, Features)

    # Apply mask only during training phase
    return tf.where(tf.equal(training, True), x * mask, x)

# 2. THE CUSTOM KERAS LAYER
class ScheduledMaskingLayer(tf.keras.layers.Layer):
    def __init__(self, prob_var, **kwargs):
        super(ScheduledMaskingLayer, self).__init__(**kwargs)
        self.prob_var = prob_var

    def call(self, inputs, training=None):
        return scheduled_mask_layer(inputs, training=training, prob=self.prob_var)

    def get_config(self):
        config = super().get_config()
        config.update({"prob_var": self.prob_var})
        return config

# 3. RE-INITIALIZE GLOBAL VARIABLE (Ensure it's a tf.Variable for Graph compatibility)
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32, name="sampling_prob_v5")

# Scheduled masking logic from your notebook
class ScheduledSamplingCallback(tf.keras.callbacks.Callback):
    def __init__(self, prob_var, decay_rate=0.05, min_prob=0.1):
        super().__init__()
        self.prob_var = prob_var      # The tf.Variable tracking the sampling probability
        self.decay_rate = decay_rate  # How much to reduce teacher forcing each epoch
        self.min_prob = min_prob      # The floor value (minimum teacher forcing retained)

    def on_epoch_end(self, epoch, logs=None):
        # 1. Retrieve the current ratio from the TensorFlow variable
        current_val = self.prob_var.numpy()
        
        # 2. Linear Decay: Calculate the new value for the next epoch
        new_val = max(self.min_prob, current_val - self.decay_rate)
        
        # 3. Update the backend variable so the Lambda layers use the new value
        tf.keras.backend.set_value(self.prob_var, new_val)
        
        # 4. Log for monitoring convergence behavior
        print(f"\n --- End of Epoch {epoch + 1}: Teacher Forcing Ratio set to {new_val:.2f} ---")

In [45]:
def train_warmup(full_model, visual_encoder, x_train, y_train, val_data, epochs=5, learning_rate=1e-3):
    visual_encoder.trainable = False
    
    # Use AdamW as requested
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    
    # Crucial: Compile after setting trainable=False
    full_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Add your sampling callback to handle the global variable during training
    sampling_cb = ScheduledSamplingCallback(sampling_prob, decay_rate=0.04, min_prob=0.5)
    
    history = full_model.fit(
        x=x_train, 
        y=y_train, 
        validation_data=val_data, 
        epochs=epochs, 
        batch_size=32,
        callbacks=[sampling_cb]
    )
    return history
import tensorflow as tf

def train_fine_tune(full_model, visual_encoder, x_train, y_train, val_data, epochs=50, learning_rate=1e-5):
    """
    Stage 2: End-to-End Fine-tuning.
    Unfreezes the Visual Encoder and uses a low learning rate to refine the 
    weights across the entire pipeline.
    """
    print("Starting Fine-tuning Stage: Unfreezing Visual Encoder...")
    
    # 1. Unfreeze the visual encoder layers
    visual_encoder.trainable = True
    
    # 2. Re-compile the model. 
    # Using AdamW (Decoupled Weight Decay) to improve generalization [2].
    # The low learning rate prevents 'Catastrophic Forgetting' [3].
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate, 
        weight_decay=1e-3
    )
    
    full_model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    
    # 3. Define Callbacks for convergence monitoring
    # Patience is higher here because fine-tuning changes are subtle.
    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10, 
        restore_best_weights=True,
        verbose=1
    )
    
    # Scheduled Sampling Callback must be included to continue decaying 
    # the teacher forcing ratio during fine-tuning [4].
    sampling_cb = ScheduledSamplingCallback(
        sampling_prob, 
        decay_rate=0.02, 
        min_prob=0.1
    )
    
    # 4. Execute training
    history = full_model.fit(
        x=x_train,
        y=y_train,
        validation_data=val_data,
        epochs=epochs,
        batch_size=32,
        callbacks=[early_stopper, lr_scheduler, sampling_cb]
    )
    
    return history

### Data

In [34]:
# Creating visual encoder training data, including ground-truth sequences used for teacher forcing.
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [35]:
# Calculator model data
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [36]:
# Full pipeline training data
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

## Full model

In [37]:
from tensorflow.keras.saving import load_model

# Mapping the serialized names to current objects
custom_objects = {
    "scheduled_mask_layer": scheduled_mask_layer,
    "sampling_prob": sampling_prob
}

# Reloading ensures the Functional Graph is reconstructed with correct references
visual_encoder = load_model('visual_encoder.keras', custom_objects=custom_objects, safe_mode=False)
calculator = load_model('calculator.keras', custom_objects=custom_objects, safe_mode=False)

In [38]:
from tensorflow.keras import Model

def build_full_model(visual_encoder, calculator):
    img_input = Input(shape=(5, 28, 28, 1), name="img_input") 
    expr_tf_input = Input(shape=(6, 15), name="expr_tf_input") 
    ans_tf_input = Input(shape=(4, 15), name="ans_tf_input")   

    # Explicitly calling the models ensures the 'training' argument 
    # reaches the internal Lambda layers
    predicted_expression = visual_encoder([img_input, expr_tf_input])
    final_output = calculator([predicted_expression, ans_tf_input])

    return Model(inputs=[img_input, expr_tf_input, ans_tf_input], outputs=final_output)

# Re-construct
model_full = build_full_model(visual_encoder, calculator)

In [39]:
import numpy as np

# 1. Expand dimensions for the grayscale channel (required by ConvLayers)
X_train_full = X_train[..., np.newaxis]
X_val_full = X_val[..., np.newaxis]

# 2. Group the inputs based on your provided variables
# Note: y_train_in_pt (from block 1) matches X_train (from block 3) due to shared random_state
train_inputs = [X_train_full, y_train_in_pt, y_train_in]
val_inputs = [X_val_full, y_val_in_pt, y_val_in]

# 3. Targets are the shifted answer sequences
train_targets = y_train_target
val_targets = y_val_target

In [40]:
import tensorflow as tf

# 1. Define the variable in the current session
# Using a tf.Variable ensures it persists in the TensorFlow graph
if 'sampling_prob' not in globals():
    sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32)

# 2. Define the function exactly as it was in the first notebook
def scheduled_mask_layer(x, training=None):
    if training:
        # Use the variable defined above
        mask = tf.cast(tf.random.uniform(tf.shape(x)[:-1]) < sampling_prob, dtype=tf.float32)
        return x * tf.expand_dims(mask, -1)
    return x

# 3. CRITICAL STEP: Manually patch the loaded models
# We iterate through the layers to find the Lambda layer and re-bind its function
def patch_model_lambdas(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Lambda):
            # We point the Lambda's function to our new local version
            layer.function = scheduled_mask_layer
    return model

# Apply the patch to your loaded models
visual_encoder = patch_model_lambdas(visual_encoder)
calculator = patch_model_lambdas(calculator)

# Now rebuild the full model
model_full = build_full_model(visual_encoder, calculator)

In [41]:


def run_full_pipeline_training(model_full, visual_encoder):
    # --- STAGE 1: WARMUP (Feature Extraction) ---
    # We freeze the encoder to protect pre-trained weights from high initial gradients
    visual_encoder.trainable = False
    
    print("Starting Stage 1: Warmup (Visual Encoder Frozen)...")
    history_warmup = train_warmup(
        full_model=model_full,
        visual_encoder=visual_encoder,
        x_train=train_inputs,
        y_train=train_targets,
        val_data=(val_inputs, val_targets),
        learning_rate=4.0e-4 # High rate for the new calculator layers
    )

    # --- STAGE 2: FINE-TUNING (End-to-End) ---
    # We unfreeze to allow the encoder to specialize for the calculator task
    visual_encoder.trainable = True
    
    print("\nStarting Stage 2: Fine-Tuning (End-to-End)...")
    history_fine_tune = train_fine_tune(
        full_model=model_full,
        visual_encoder=visual_encoder,
        x_train=train_inputs,
        y_train=train_targets,
        val_data=(val_inputs, val_targets),
        learning_rate=1.0e-5 # Low rate to prevent Catastrophic Forgetting
    )
    
    return history_warmup, history_fine_tune

# Run the training
# Ensure model_full is built using your build_full_model(visual_encoder, calculator)
warmup_h, finetune_h = run_full_pipeline_training(model_full, visual_encoder)

Starting Stage 1: Warmup (Visual Encoder Frozen)...
Epoch 1/5
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7322 - loss: 0.9197
 --- End of Epoch 1: Teacher Forcing Ratio set to 0.96 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.8122 - loss: 0.6452 - val_accuracy: 0.7777 - val_loss: 0.9077
Epoch 2/5
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9063 - loss: 0.3795
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.92 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9107 - loss: 0.3702 - val_accuracy: 0.7906 - val_loss: 0.8750
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9189 - loss: 0.3458
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.88 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9208 - loss: 0.3360 - val_accuracy: 0.7872 - val_loss: 0.8709
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9246 - loss: 0.3254
 --- End of Epoch 4: Teacher Forcing Ratio set to 0.84 ---
500/500 ━━━

E0000 00:00:1767879898.558172 3369405 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_132/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_132/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_948/gradient_tape/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/functional_13_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.9358 - loss: 0.2855
 --- End of Epoch 1: Teacher Forcing Ratio set to 0.78 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - categorical_accuracy: 0.9382 - loss: 0.2754 - val_categorical_accuracy: 0.8191 - val_loss: 0.7387
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.9413 - loss: 0.2645
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.76 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 54ms/step - categorical_accuracy: 0.9422 - loss: 0.2606 - val_categorical_accuracy: 0.8040 - val_loss: 0.7918
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.9425 - loss: 0.2564
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.74 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 54ms/step - categorical_accuracy: 0.9428 - loss: 0.2551 - val_categorical_accuracy: 0.8256 - val_loss: 0.7061
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.9443 - loss: 0.2495
 --- End 

In [47]:
history_fine_tune = train_fine_tune(
        full_model=model_full,
        visual_encoder=visual_encoder,
        x_train=train_inputs,
        y_train=train_targets,
        val_data=(val_inputs, val_targets)
    )

Starting Fine-tuning Stage: Unfreezing Visual Encoder...
Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - categorical_accuracy: 0.9568 - loss: 0.1944
 --- End of Epoch 1: Teacher Forcing Ratio set to 0.48 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - categorical_accuracy: 0.9561 - loss: 0.1969 - val_categorical_accuracy: 0.8441 - val_loss: 0.6442 - learning_rate: 1.0000e-05
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - categorical_accuracy: 0.9580 - loss: 0.1932
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.46 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 53ms/step - categorical_accuracy: 0.9563 - loss: 0.1968 - val_categorical_accuracy: 0.8481 - val_loss: 0.6361 - learning_rate: 1.0000e-05
Epoch 3/50
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - categorical_accuracy: 0.9568 - loss: 0.1965
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.44 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 53ms/step - categorical_accuracy: 0.9562 - loss: 0.1994 - val_categorical_accuracy: 0.8430 - v

# Inference

In [48]:
import numpy as np

def run_full_inference(visual_encoder, calculator, X_img_test, char_to_index):
    """
    Performs end-to-end inference from images to mathematical result.
    
    1. Visual Encoder: Images -> Predicted Expression
    2. Calculator: Predicted Expression -> Final Answer
    """
    # Inverse mapping for decoding
    index_to_char = {v: k for k, v in char_to_index.items()}
    num_samples = X_img_test.shape[0]
    
    # 1. Prepare Image Input (Ensuring 5D shape [N, 5, 28, 28, 1])
    X_test_expanded = X_img_test[..., np.newaxis]
    
    print(f"Starting inference on {num_samples} samples...")
    
    # --- STAGE 1: VISUAL ENCODING ---
    # We pass a dummy/zero tensor for teacher forcing because visual_encoder 
    # internal logic handles inference when training=False
    dummy_expr = np.zeros((num_samples, 6, 15)) 
    # Use training=False to ensure Lambda layers (Scheduled Sampling) are bypassed [3]
    pred_expressions_dist = visual_encoder.predict([X_test_expanded, dummy_expr], verbose=0)
    
    # --- STAGE 2: CALCULATION ---
    # The calculator also expects [expression, answer_tf]
    dummy_ans = np.zeros((num_samples, 4, 15))
    pred_answers_dist = calculator.predict([pred_expressions_dist, dummy_ans], verbose=0)
    
    # --- STAGE 3: DECODING ---
    results = []
    for i in range(num_samples):
        # Convert probability distributions to character sequences via Argmax [4]
        expr_indices = np.argmax(pred_expressions_dist[i], axis=-1)
        ans_indices = np.argmax(pred_answers_dist[i], axis=-1)
        
        expr_str = "".join([index_to_char[idx] for idx in expr_indices])
        ans_str = "".join([index_to_char[idx] for idx in ans_indices])
        
        results.append({
            "expression": expr_str.replace('<PAD>', '').replace('<EOS>', ''),
            "answer": ans_str.replace('<PAD>', '').replace('<EOS>', '')
        })
        
    return results

# Usage:
# results = run_full_inference(visual_encoder, calculator, X_test, char_to_index)

In [57]:
import numpy as np

def calculate_full_model_metrics(results, y_test_target, char_to_index):
    """
    Calculates accuracy metrics for the full visual calculator pipeline.
    
    results: List of dicts from the run_full_inference loop [{'expression': str, 'answer': str}, ...]
    y_test_target: The ground truth answers (one-hot, shape [N, 4, 15])
    char_to_index: Dictionary for inverse mapping
    """
    index_to_char = {v: k for k, v in char_to_index.items()}
    num_samples = len(results)
    
    total_tokens = 0
    correct_tokens = 0
    correct_math_results = 0
    
    # 1. Decode Ground Truth Answers
    y_true_indices = np.argmax(y_test_target, axis=-1)
    
    for i in range(num_samples):
        # Clean Ground Truth string
        true_ans_str = "".join([index_to_char[idx] for idx in y_true_indices[i]])
        true_ans_clean = true_ans_str.replace('<PAD>', '').replace('<EOS>', '')
        
        # Predicted Answer from inference loop
        pred_ans_clean = results[i]['answer']
        
        # --- A. Math Accuracy (Exact Match) ---
        # Checks if the entire sequence is identical to the target [2]
        if pred_ans_clean == true_ans_clean:
            correct_math_results += 1
            
        # --- B. Token Accuracy (Character Level) ---
        # We compare character by character up to the length of the shorter string [3]
        # (Alternatively, you can pad to fixed length for strict comparison)
        min_len = min(len(true_ans_clean), len(pred_ans_clean))
        max_len = max(len(true_ans_clean), len(pred_ans_clean))
        
        for char_idx in range(min_len):
            if true_ans_clean[char_idx] == pred_ans_clean[char_idx]:
                correct_tokens += 1
        
        total_tokens += max_len
    
    # 2. Compute Percentages
    math_accuracy = (correct_math_results / num_samples) * 100
    token_accuracy = (correct_tokens / total_tokens) * 100
    
    print(f"--- Full Pipeline Evaluation ---")
    print(f"Math Accuracy (Exact Sequence Match): {math_accuracy:.2f}%")
    print(f"Token Accuracy (Character Level):     {token_accuracy:.2f}%")
    
    return math_accuracy, token_accuracy

# Usage:
# results = run_full_inference(visual_encoder, calculator, X_test, char_to_index)
# math_acc, token_acc = calculate_full_model_metrics(results, y_test_target, char_to_index)

In [59]:
results = run_full_inference(visual_encoder, calculator, X_test, indices)
math_acc, token_acc = calculate_full_model_metrics(results, y_test_target, indices)

Starting inference on 2000 samples...
--- Full Pipeline Evaluation ---
Math Accuracy (Exact Sequence Match): 20.15%
Token Accuracy (Character Level):     86.28%
